In [1]:
import requests
import pandas as pd
import time
import re
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
torch.manual_seed(42)

In [18]:
def create_model():
    model_name = "t-tech/T-lite-it-1.0"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
    )

    return model, tokenizer

In [4]:
df = pd.read_csv('ru_cefr_short.csv')
df

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [6]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

In [7]:
role = 'Ты — экспертный ассистент-лингвист, специализирующийся на анализе уровня сложности текстов на русском языке. Твоя основная задача — помогать менять текст, сохраняя его уровень сложности, опираясь на лексические, грамматические, синтаксические и стилистические признаки. Ты действуешь как профессиональный лингвист, методист по русскому языку и специалист по оценке текстовой сложности. Если тебе нужно преобразовать текст, ориентируйся на то, чтобы сохранять исходный смысл, но при этом точно сохранить уровень сложности языка. Ты особенно внимателен к естественности русского языка. Не допускай неестественных формулировок, канцелярита без необходимости, ломаного стиля, кальки с других языков и механических замен слов ради сложности. Все изменения должны выглядеть так, как будто текст написал носитель русского языка с соответствующим уровнем речевой компетенции. Твоя цель — не просто менять текст, а лингвистически обоснованно сохранять его сложность на русском языке.'

# 3. Изменение порядка предложений

In [7]:
def create_prompt(text):
    return f'''Перепиши этот текст, изменив порядок предложений или частей предложений, сохраняя исходный смысл и структуру. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: "Исследователи работают над системами, способными анализировать дорогу лучше, чем водитель-человек."

Аугментация: "Системы, способные анализировать дорогу лучше, чем человек, сейчас разрабатывают исследователи."
    
Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:'''

In [8]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Радиация сама по себе незаразна; например, её мощные дозы от изотопов часто используют для стерилизации фруктов и овощей, а также для лечения людей. (3:38–3:54)

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Waymo запустила приложение на Android для владельцев своих самоуправляемых машин, а недавно Uber объявила об инвестиции в размере одного миллиарда долларов в свою систему автоматизированного вождения.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика,

In [9]:
df_pred.to_csv("c2_augmented_3_t_lite.csv")

# 4. Реорганизация фраз

In [10]:
def create_prompt(text):
    return f'''Перепиши этот текст, изменив порядок слов в предложении, сохраняя исходный смысл и структуру. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: "Бедный, так и не выпьешь нормально. Не расслабишься."


Аугментация: "Не расслабишься, бедный, не выпьешь нормально."
    
Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:'''

In [11]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Мощными дозами излучения, от тех же самых изотопов, часто стерилизуют фрукты и овощи, а также лечат людей; собственной радиацией она незаразна (3:38–3:54).

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Waymo даже запустила приложение на Android для владельцев своих самоуправляемых машин, а недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика, дн

In [12]:
df_pred.to_csv("c2_augmented_4_t_lite.csv")

# 5. Усложнение объяснений

In [13]:
def create_prompt(text):
    return f'''Перепиши этот текст, сделав объяснения более детальными и развернутыми. Добавь дополнительные пояснения, уточнения, примеры или расширенные определения ключевых понятий.
Пример:

Оригинал: "В мире обострилась конкуренция за таланты: их переманивают высокими зарплатами..."

Аугментация: "В мире возникла жесткая борьба за таланты. Люди меняют работу ради высоких зарплат и перспектив..."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:'''

In [14]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	На данном временном отрезке, с 3:38 до 3:54, подчеркивается, что радиация сама по себе не является заразной. Это означает, что она не передается от одного объекта к другому через прямой контакт или воздушно-капельным путем. Для иллюстрации этого принципа можно привести пример: мощные дозы излучения, исходящие от определенных изотопов, широко используются в процессе стерилизации фруктов и овощей. Этот метод позволяет уничтожить микроорганизмы, обеспечивая безопасность продуктов питания для потребителей. Аналогичным образом, радиационная терапия применяется в медицине для лечения различных заболеваний, таких как рак. В ходе этой процедуры используется целенаправленное воздействие ионизирующего излучения на пораженные клетки, что способствует их унич

In [15]:
df_pred.to_csv("c2_augmented_5_t_lite.csv")

# 8. Использование аналогий

In [16]:
def create_prompt(text):
    return f'''Перепиши следующий текст, используя аналогию для объяснения основного понятия. Используй такую аналогию, которая сделает текст более наглядным, но при этом сохранит высокий уровень сложности и не упростит содержание. Аналогия должна быть метафорической и соответствовать интеллектуальному уровню оригинала. Важно, чтобы аналогия не заменила ключевое содержание текста, а лишь помогла лучше донести его суть.

Пример:

Оригинал: "В мире обострилась конкуренция за таланты..."

Аугментация: "Конкуренция за таланты теперь как гонка за редким ресурсом, который необходим каждому высокотехнологичному бизнесу."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:'''

In [17]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Радиация подобна невидимой реке, текущей сквозь продукты и человеческие тела: она не передается от одного к другому, но её течение может быть использовано для стерилизации, подобно тому, как вода используется для очищения, или для лечения, как врач применяет лекарство.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Автоматизация вождения стала для компаний, как стратегическая крепость в шахматах, где Uber и Waymo борются за контроль над будущим мобильности, вкладывая миллиарды в свои системы, подобно тому, как игроки вкладывают все сил

In [18]:
df_pred.to_csv("c2_augmented_8_t_lite.csv")

# 9. Синтаксические трансформации

In [19]:
def create_prompt(text):
    return f'''Перепиши этот текст, заменяя активные грамматические конструкции на пассивные, а пассивные — на активные там, где это возможно и стилистически уместно. Сохраняй исходный смысл, терминологию и общий уровень сложности текста. Обрати внимание на сохранение логических связей между частями предложений.
Пример:

Оригинал: "Множество повестей, рассказов, публицистики, дневников были написаны великим автором."

Аугментация: "Великим автором было написано множество повестей, рассказов, публицистики и дневников."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:'''

In [20]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Радиация сама по себе не является заразной. Например, мощные дозы излучения от изотопов часто используют для стерилизации фруктов и овощей, а также для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила об инвестировании одного миллиарда долларов в свою систему автоматизированного вождения, а компания Waymo запустила приложение на Android для владельцев своих самоуправляемых машин.  
  
Переписанный вариант:  
Компания Uber объявила об инвестициях в размере одного миллиарда долларов в собственную систе

In [21]:
df_pred.to_csv("c2_augmented_9_t_lite.csv")

# 11. Перефразирование

In [22]:
def create_prompt(text):
    return f"""Перефразируй этот текст на русском языке, сохраняя его уровень сложности. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: 'Искусственный интеллект меняет способы взаимодействия людей с технологиями.'

Аугментация: 'Технологии искусственного интеллекта трансформируют методы коммуникации человека с цифровыми устройствами.'

Текст: '{text}'

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, верни только ОДНУ новую версию переписанного текста, не заключай новый текст в кавычки:"""

In [23]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Радиация сама по себе не является заразной; например, высокие дозы излучения от изотопов часто используют для стерилизации фруктов и овощей, а также в лечебных целях для людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber вложила один миллиард долларов в разработку собственной системы автономного вождения, а Waymo уже представила приложение для Android, предназначенное для пользователей их самоуправляемых автомобилей.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рас

In [24]:
df_pred.to_csv("c2_augmented_11_t_lite.csv")

# Генерация по требованиям + пример

In [8]:
def create_prompt():
    return f"""
Сгенерируй  текст на русском языке уровня сложности С2 длиной от 15 до 30 слов.

Требования к уровню С2:
Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. Уровень С2 подразумевает владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы.

Пример текста уровня С2: 
Покачиваясь вместе с вагоном, мы преодолевали бескрайние просторы Сибири, и, чем дальше мы уезжали от столиц и больших городов, тем чаще ловили себя на мысли, что сама протяжённость этого пути, постоянная смена пейзажа и наших попутчиков, становится самоцелью путешествия, расстояния измеряются здесь уже не километрами, а прожитыми мгновениями: как пелось в той песне, “есть только миг”. 

Комментарий: очень сложная структура предложения с множеством связей, сложные союзы и предлоги (чем, тем), устойчивые выражения (бескрайние просторы, ловили себя на мысли), связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов (“есть только миг”.)

Верни только новый текст на русском языке без пояснений и комментариев, верни только ОДИН новый текст, не заключай новый текст в кавычки:"""

In [11]:
df_pred = pd.DataFrame(columns=['generated-text'])
prompt = create_prompt()

print("Генерируем 120 текстов...")

for i in range(120):
    model, tokenizer = create_model()
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"{i+1}. Сгенерированный текст:\n\t{generated_text}\n")
   
    new_row = pd.DataFrame({
        'generated-text': [generated_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

Генерируем 120 текстов...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

1. Сгенерированный текст:
	Исследуя сложные механизмы современной экономики, мы обнаруживаем, что взаимосвязь между инфляцией и безработицей, представляющая собой классическую модель Адама Смита, в условиях глобализации претерпевает значительные изменения, обозначенные как "парадокс Фридмана", который ведет к неожиданным последствиям для мирового рынка.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2. Сгенерированный текст:
	Исследуя глубины космоса, учёные обнаруживают, что каждая звезда представляет собой уникальную систему, состоящую из элементов, которые взаимодействуют, образуя сложные структуры, из которых следует вывод о многообразии вселенной и её эволюции.



...

118. Сгенерированный текст:
	Исследуя динамику климатических изменений, учёные утверждают, что повышение средней глобальной температуры, обусловленное антропогенной деятельностью, ведёт к увеличению частоты экстремальных погодных явлений, таких как ураганы и засухи, что, в свою очередь, требует разработки новых стратегий адаптации и смягчения последствий.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

119. Сгенерированный текст:
	Исследуя глубины космоса, мы осознаём, что каждая звезда, которую мы видим, представляет собой уникальную систему, состоящую из множества планет, и изучение их атмосфер ведёт к пониманию происхождения жизни, из чего можно сделать вывод о многообразии форм существования во Вселенной.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

120. Сгенерированный текст:
	Исследуя дебри Амазонии, учёные обнаружили, что экосистема, состоящая из разнообразных видов флоры и фауны, представляет собой сложную сеть взаимосвязей, где каждый элемент играет важную роль, из чего следует вывод о необходимости её защиты и сохранения.



In [12]:
lengths = [len(text.split()) for text in df_pred['generated-text']]
min(lengths), max(lengths)

(26, 89)

In [13]:
df_pred.to_csv("c2_generated_with_example_t_lite.csv")

# Генерация по требованиям + без примеров

In [14]:
def create_prompt():
    return f"""
Сгенерируй  текст на русском языке уровня сложности С2 длиной от 15 до 30 слов.

Требования к уровню С2:
Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. Уровень С2 подразумевает владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Верни только новый текст на русском языке без пояснений и комментариев, верни только ОДИН новый текст, не заключай новый текст в кавычки:"""

In [19]:
df_pred = pd.DataFrame(columns=['generated-text'])
prompt = create_prompt()

print("Генерируем 120 текстов...")

for i in range(120):
    model, tokenizer = create_model()
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"{i+1}. Сгенерированный текст:\n\t{generated_text}\n")
   
    new_row = pd.DataFrame({
        'generated-text': [generated_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

Генерируем 120 текстов...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

1. Сгенерированный текст:
	Анализируя современные тенденции в области искусственного интеллекта, можно сделать вывод, что развитие алгоритмов машинного обучения, обладающих способностью к самообучению, ведет к значительному расширению границ автоматизации процессов в различных сферах деятельности, что, в свою очередь, обозначает собой новую эру цифровой трансформации.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2. Сгенерированный текст:
	Исследование, посвященное феномену глобального потепления, демонстрирует, что увеличение концентрации парниковых газов в атмосфере ведет к повышению средней температуры Земли, что, в свою очередь, обозначается как угроза экосистемам и человеческому здоровью. Из этого следует, что разработка и внедрение технологий возобновляемой энергии являются необходимым условием для обеспечения устойчивого развития.



...

118. Сгенерированный текст:
	Анализируя современные тенденции в области искусственного интеллекта, следует отметить, что развитие нейронных сетей, представляющих собой сложные алгоритмы машинного обучения, ведет к созданию более точных систем распознавания образов, которые характеризуются способностью обрабатывать огромные объемы данных и извлекать из них ценные знания, что, в свою очередь, обозначается как значительный прогресс в научных исследованиях и технологическом прогрессе.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

119. Сгенерированный текст:
	Исследование показывает, что инновационные технологии, обозначаемые как "умные" системы, представляют собой комплексное решение, состоящее из искусственного интеллекта и Big Data, которые, в свою очередь, ведут к повышению эффективности бизнес-процессов и позволяют сделать вывод о необходимости их внедрения в современную экономику.



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

120. Сгенерированный текст:
	Исследование показывает, что инновационные технологии, обозначаемые как "цифровая трансформация", представляют собой процесс, который ведет к повышению эффективности бизнеса, состоящий из внедрения искусственного интеллекта и облачных вычислений, из чего можно сделать вывод о значительном влиянии на современные рыночные условия.



In [20]:
lengths = [len(text.split()) for text in df_pred['generated-text']]
min(lengths), max(lengths)

(25, 57)

In [21]:
df_pred.to_csv("c2_generated_t_lite.csv")

# Аугментация В2

In [22]:
texts_to_augment_b2 = []

for i in range(len(train_labels)):
  if train_labels[i] == 4:
    texts_to_augment_b2.append(train_texts[i])

texts_to_augment_b2 = texts_to_augment_b2[:120]
len(texts_to_augment_b2)

120

In [23]:
def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [24]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment_b2)} текстов...")

for text in texts_to_augment_b2:
    model, tokenizer = create_model()
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к дому невесты его встретила серия неожиданных препятствий, поскольку друзья и подруги невесты, стремясь обеспечить соблюдение традиций, отказывались пропустить его без того, чтобы он не сделал какой-либо дар или не возместил их расходы, что, по сути, являлось необходимым условием для продолжения торжества.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Наши исследования продемонстрировали, что продолжительность жизни представителей творческих профессий на протяжении всех исторических периодов существенно превышает среднестатистическую продолжительность жизни, что обусловлено, в частности, их способностью к адаптации и высокой устойчивостью к стрессам. Это обстоятельство, безусловно, свидетельствует о том, что творческие индивиды обладают уникальными характеристиками, которые позволяют им эффективно справляться с жизненными вызовами, и, следовательно, из этого можно сделать вывод о важности поддержания и развития творческих способностей в обществе.


...


118. Реальный текст:
	Постоянно вижу припаркованные на тротуарах машины во дворах многоэтажек. Мне это надоело, и я решила с этим бороться.
Аугметированный текст:
	Наблюдая за регулярным явлением парковки транспортных средств на тротуарах в придомовых территориях многоэтажных домов, я испытываю чувство раздражения, что побуждает меня к активному противодействию этой практике, поскольку она создает неудобства для жителей и нарушает общественный порядок.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


119. Реальный текст:
	По площади она в два раза больше Соединённых Штатов Америки, но в два раза меньше по населению. В России живёт примерно 145 миллионов человек.
Аугметированный текст:
	Распространённая территория Российской Федерации превосходит по размерам Соединённые Штаты Америки вдвое, однако её численность населения составляет около 145 миллионов человек, что вдвое меньше, чем в США. Это обстоятельство обусловливает уникальную демографическую динамику страны, где обширные просторы контрастируют с плотностью населения. Можно сделать вывод, что Россия, будучи обширной страной, характеризуется низкой плотностью населения, что ведёт к специфическим социально-экономическим условиям и вызовам.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


120. Реальный текст:
	Дом, кухня, сад — всё это напоминало деревенскую усадьбу, и это было очень по душе всем членам многочисленной семьи Л. Н. Толстого.
Аугметированный текст:
	Все элементы — от просторного дома до ухоженного сада — создавали атмосферу типичной деревенской усадьбы, что вызывало особое удовлетворение у всех членов многочисленной семьи Льва Николаевича Толстого, поскольку каждый из них находил здесь отражение своих самых глубинных воспоминаний и традиций. Это пространство, обозначенное как место духовного и семейного единства, представляло собой нечто большее, чем просто жилище; оно символизировало гармонию быта и природы, из которой следовало понимание истинных ценностей жизни.


In [25]:
df_pred.to_csv("c2_from_b2_augmented_t_lite.csv")

# Аугментация С1

In [26]:
texts_to_augment_c1 = []

for i in range(len(train_labels)):
  if train_labels[i] == 5:
    texts_to_augment_c1.append(train_texts[i])

texts_to_augment_c1 = texts_to_augment_c1[:120]
len(texts_to_augment_c1)

120

In [27]:
def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [28]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment_c1)} текстов...")

for text in texts_to_augment_c1:
    model, tokenizer = create_model()
    prompt = create_prompt(text)
    messages = [
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    question = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([question], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    augmented_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	В глубине моего восприятия возникло осознание: передо мной находится моя супруга. Её лицо, отмеченное тонкой белизной, озарено глазами, крупными, выпуклыми, черными, — они словно отражают мою собственную сущность, мою внутреннюю душу.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Создается парадоксальная дилемма: индивид, наделенный досугом и потенциалом радикально трансформировать собственную судьбу, погружается в самодостаточность, отгораживаясь от внешнего мира.


...


118. Реальный текст:
	Любой квалифицированный сотрудник сейчас имеет зарплату, явно перекрывающую его основные нужды (питание, одежда и т. д.) минимум втрое.
Аугметированный текст:
	В современных условиях любой специалист обладает заработной платой, которая, безусловно, значительно превышает его базовые потребности, такие как питание и одежда, по крайней мере в три раза, что свидетельствует о высоком уровне материальной обеспеченности.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


119. Реальный текст:
	Мне кажется, что вот это москвичам удавалось лучше; впрочем, я могу быть пристрастен — ведь я сам стихотворный переводчик московской школы!
Аугметированный текст:
	Считаю, что москвичи демонстрировали более высокий уровень мастерства в этой области, однако признаю возможность субъективности моего мнения, поскольку являюсь представителем московской школы стихотворных переводов.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


120. Реальный текст:
	Вот вы облились. А деньги перевели?» (3) Впрочем, большинство пользователей считают, что акция принесла большую пользу.
Аугметированный текст:
	Учитывая инцидент с попаданием жидкости на одежду, возникает вопрос о своевременности осуществления денежных переводов. Тем не менее, подавляющее большинство участников акции придерживаются мнения, что она оказала значительное положительное воздействие на их опыт.


In [29]:
df_pred.to_csv("c2_from_c1_augmented_t_lite.csv")